# ĐỀ TÀI: HỆ THỐNG NHẬN DẠNG NGƯỜI NÓI (SPEAKER RECOGNITION) TỪ GIỌNG NÓI SỬ DỤNG MACHINE LEARNING

## Vấn đề thực tiễn
Trong thời đại số, nhận dạng giọng nói (Speaker Recognition) là công nghệ sinh trắc học quan trọng, được ứng dụng rộng rãi trong bảo mật tài khoản, trợ lý ảo thông minh, kiểm soát truy cập hệ thống và pháp y. Giọng nói của mỗi người là đặc trưng sinh lý độc đáo (thanh quản, vòm họng, lưỡi). Việc tự động nhận diện người nói từ file âm thanh giúp thay thế mật khẩu truyền thống, tăng tính tiện lợi và an toàn.

## Mục tiêu
- Tiền xử lý dữ liệu âm thanh thô (cắt đoạn 1 giây, loại bỏ khoảng lặng).
- Trích xuất đặc trưng MFCC siêu cấp (240 chiều: MFCC + Delta + Delta2).
- Huấn luyện và so sánh 2 mô hình: SVM (Support Vector Machine) và KNN (K-Nearest Neighbors).
- Xây dựng hệ thống dự đoán danh tính người nói thời gian thực.
- Phân tích độ chính xác, lưu mô hình và đưa ra khuyến nghị ứng dụng thực tế.

## Dữ liệu
- **Nguồn**: Thư mục `data/raw/` (các thư mục con mang tên người nói, chứa file `.wav` gốc dài).
- **Sau xử lý**: Thư mục `data/processed/` (tự động cắt thành các mẫu 1 giây, đã lọc khoảng lặng).
- **Đặc trưng cuối cùng**: Vector 240 chiều cho mỗi mẫu âm thanh.

## Công cụ và thư viện
- **Xử lý âm thanh**: librosa, soundfile.
- **Trích xuất đặc trưng**: MFCC + Delta (librosa).
- **Machine Learning**: scikit-learn (SVC, KNeighborsClassifier, StandardScaler, train_test_split).
- **Lưu trữ mô hình**: pickle.
- **Khác**: numpy, tqdm, os, pandas.
- **Cấu trúc dự án**: Giống hệ thống module (1_preprocess → 2_train → 3_predict).

---

## 1. Cài đặt các thư viện cần thiết

In [1]:
import sys
import subprocess

# Danh sách thư viện cần thiết
packages = [
    'librosa',
    'soundfile',
    'scikit-learn',
    'numpy',
    'tqdm',
    'pandas'
]

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} đã được cài đặt")
    except ImportError:
        print(f"⚠ Đang cài đặt {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} đã cài đặt thành công")



## 2. Import thư viện

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import pickle
import warnings
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

PROCESSED_DATA_DIR = "data/processed"
MODEL_DIR = "models"
TARGET_SR = 16000

print("✓ Tất cả thư viện đã được import thành công!")



## 3. Tiền xử lý dữ liệu âm thanh (Cắt 1 giây + Loại bỏ khoảng lặng)


In [3]:
RAW_DATA_DIR = "data/raw"
CHUNK_DURATION = 1.0  # cắt thành đoạn 1 giây

def process_person_folder(person_name):
    person_raw_dir = os.path.join(RAW_DATA_DIR, person_name)
    person_processed_dir = os.path.join(PROCESSED_DATA_DIR, person_name)
    os.makedirs(person_processed_dir, exist_ok=True)

    chunk_count = 1
    print(f"\n[*] Đang xử lý dữ liệu của: {person_name}")

    files = [f for f in os.listdir(person_raw_dir) if f.endswith(".wav")]
    for file_name in tqdm(files, desc="Đang cắt file", unit="file"):
        file_path = os.path.join(person_raw_dir, file_name)
        try:
            y, sr = librosa.load(file_path, sr=TARGET_SR)
            y_trimmed, _ = librosa.effects.trim(y, top_db=30)  # loại bỏ khoảng lặng
            chunk_samples = int(CHUNK_DURATION * sr)
            total_samples = len(y_trimmed)

            for start_i in range(0, total_samples, chunk_samples):
                end_i = start_i + chunk_samples
                chunk = y_trimmed[start_i:end_i]
                if len(chunk) == chunk_samples:
                    output_path = os.path.join(person_processed_dir, f"{chunk_count}.wav")
                    librosa.output.write_wav(output_path, chunk, sr)  # hoặc dùng soundfile
                    chunk_count += 1
        except Exception as e:
            print(f"[-] Lỗi file {file_name}: {e}")

    print(f"  -> Đã tạo {chunk_count - 1} mẫu âm thanh (1 giây/mẫu)")

# Chạy tiền xử lý
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

for item in os.listdir(RAW_DATA_DIR):
    person_dir = os.path.join(RAW_DATA_DIR, item)
    if os.path.isdir(person_dir):
        process_person_folder(item)

print("\n[+] Hoàn tất tiền xử lý dữ liệu!")



## 4. Trích xuất đặc trưng MFCC (240 chiều) & Huấn luyện mô hình


In [ ]:
def extract_mfcc(file_path):
    try:
        y, sr = librosa.load(file_path, sr=TARGET_SR)
        if len(y) < sr * 0.25:
            return None

        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        delta_mfccs = librosa.feature.delta(mfccs)
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)

        combined_mfccs = np.vstack((mfccs, delta_mfccs, delta2_mfccs))
        mfccs_mean = np.mean(combined_mfccs.T, axis=0)
        mfccs_std = np.std(combined_mfccs.T, axis=0)

        return np.hstack((mfccs_mean, mfccs_std))
    except:
        return None

# Load dữ liệu
X, y = [], []
print("[*] Đang trích xuất 240 đặc trưng MFCC...")

for person_name in os.listdir(PROCESSED_DATA_DIR):
    person_dir = os.path.join(PROCESSED_DATA_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue
    files = [f for f in os.listdir(person_dir) if f.endswith(".wav")]
    for file_name in tqdm(files, desc=f"Xử lý {person_name}"):
        features = extract_mfcc(os.path.join(person_dir, file_name))
        if features is not None:
            X.append(features)
            y.append(person_name)

X = np.array(X)
y = np.array(y)
print(f"\n[+] Dữ liệu sẵn sàng: {X.shape[0]} mẫu × 240 đặc trưng.")

if len(X) == 0:
    print("[-] Không có mẫu dữ liệu nào. Vui lòng chạy tiền xử lý trước.")
else:
    print("[*] Đang lưu toàn bộ ma trận đặc trưng ra file CSV...")
    df_mfcc = pd.DataFrame(X)
    df_mfcc['Label'] = y
    csv_filename = "mfcc_features_dataset.csv"
    df_mfcc.to_csv(csv_filename, index=False)
    print(f"[+] '{csv_filename}' để xem.\n")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("Training SVM...")
    svm_model = SVC(kernel='linear', probability=True)
    svm_model.fit(X_train_scaled, y_train)

    #  SVM
    svm_y_pred = svm_model.predict(X_test_scaled)
    print(f"Độ chính xác SVM (Accuracy): {accuracy_score(y_test, svm_y_pred) * 100:.2f}%\n")
    print("SVM: Confusion Matrix ")
    print(confusion_matrix(y_test, svm_y_pred))
    print("\nSVM")
    print(classification_report(y_test, svm_y_pred))

    print("\nTraining KNN...")
    knn_model = KNeighborsClassifier(n_neighbors=3)
    knn_model.fit(X_train_scaled, y_train)

    # KNN
    knn_y_pred = knn_model.predict(X_test_scaled)
    print(f"Độ chính xác KNN (Accuracy): {accuracy_score(y_test, knn_y_pred) * 100:.2f}%\n")
    print(" KNN: Confusion Matrix ")
    print(confusion_matrix(y_test, knn_y_pred))
    print("\n KNN")
    print(classification_report(y_test, knn_y_pred))


    # Lưu mô hình
    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, "scaler.pkl"), 'wb') as f: pickle.dump(scaler, f)
    with open(os.path.join(MODEL_DIR, "svm_model.pkl"), 'wb') as f: pickle.dump(svm_model, f)
    with open(os.path.join(MODEL_DIR, "knn_model.pkl"), 'wb') as f: pickle.dump(knn_model, f)

    print(f"\n[+] Mô hình đã được lưu tại thư mục: {MODEL_DIR}")



## 5. Dự đoán danh tính người nói (Inference)
*(Tương đương file `3_predict.py` – Test với file `data/processed/recording.wav`)*

In [ ]:
def predict_with_both_models(audio_path):
    # Load mô hình
    with open(os.path.join(MODEL_DIR, "scaler.pkl"), 'rb') as f: scaler = pickle.load(f)
    with open(os.path.join(MODEL_DIR, "svm_model.pkl"), 'rb') as f: svm_model = pickle.load(f)
    with open(os.path.join(MODEL_DIR, "knn_model.pkl"), 'rb') as f: knn_model = pickle.load(f)

    features = extract_mfcc(audio_path)
    if features is None:
        print("[-] File quá ngắn hoặc lỗi!")
        return

    np.set_printoptions(threshold=np.inf, suppress=True)
    print(features)
    print("-" * 50)

    features_scaled = scaler.transform(features.reshape(1, -1))

    svm_pred = svm_model.predict(features_scaled)[0]
    svm_prob = np.max(svm_model.predict_proba(features_scaled)) * 100

    knn_pred = knn_model.predict(features_scaled)[0]
    knn_prob = np.max(knn_model.predict_proba(features_scaled)) * 100

    print("\nKết quả dự đoán:")
    print(f"1. SVM  → {svm_pred} ({svm_prob:.2f}%)")
    print(f"2. KNN  → {knn_pred} ({knn_prob:.2f}%)")

    if svm_pred == knn_pred:
        print(f"\n[+] KẾT LUẬN: Đây là giọng của **{svm_pred}** (cả 2 mô hình đồng ý)")
    else:
        print("\n[!] Hai mô hình cho kết quả khác nhau.")

# Chạy thử
test_file = "data/processed/recording.wav"
if os.path.exists(test_file):
    predict_with_both_models(test_file)
else:
    print("[-] Vui lòng đặt file test vào data/processed/recording.wav")